In [ ]:
# INQUIRE_CSI – Contaminant Source Identification Pipeline
## Part 1 – INQUIRE_CSI_Pt1_Preprocessing
"""
Harmonizes the source databases into one master contaminant source database, removes duplicate sources, 
keeps only compounds detected in the samples, and builds one compound list per sample.
Documentation: `docs/01_preprocessing.md`
"""


In [ ]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
# Imports & setup
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import logging
import hashlib
import random
from tqdm.auto import tqdm
from tqdm.auto import trange
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import beta
import openpyxl
import time
import matplotlib.pyplot as plt
from collections import Counter

# Logging setup (equivalent to logger + appender_tee)
log_file = f"inquire_csi_pt1_preprocessing_log_{datetime.now().strftime('%y%m%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler(log_file, mode="w"),
        logging.StreamHandler()
    ]
)


# Helper functions
def majorsep():
    logging.info("=" * 100)

def minorsep():
    logging.info("-" * 100)

def percent(n, N):
    return f"{(100 * n / N):.1f}%" if N > 0 else "0%"

# Validation helpers
def check_file(path_obj, label):
    if not path_obj.exists():
        raise FileNotFoundError(f" - {label} missing: {path_obj}")
    logging.info(f" - Found file: {label}")

def check_columns(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f" - {df_name} missing required columns: {missing}")
    logging.info(f" - {df_name}: required columns present")

# Preferred encodings (tried in order)
ENCODINGS = ["utf-8", "utf-8-sig", "cp1252", "cp1250", "latin1"]

# Read only the header row using multiple encodings. Supports CSV, TSV, XLSX and XLS.
def _read_header_only(path: Path):
    suffix = path.suffix.lower()
    if suffix in [".csv", ".tsv"]:
        sep = "\t" if suffix == ".tsv" else ","
        last_error = None
        for enc in ENCODINGS:
            try:
                df = pd.read_csv(path, nrows=0, sep=sep, encoding=enc)
                logging.info(f" - Header read using encoding '{enc}'")
                return list(df.columns)
            except UnicodeDecodeError as e:
                last_error = e
        raise RuntimeError(f"Unable to decode header using supported encodings.\nLast error:\n{last_error}")
    elif suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(path, nrows=0)
        return list(df.columns)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

# Check that all expected columns exist before loading the full dataset
def check_cols_before_load(path, expected_cols, dataset_name="dataset"):
    path = Path(path)
    try:
        cols = _read_header_only(path)
    except Exception as e:
        raise RuntimeError(f"[{dataset_name}] failed to read header:\n{e}")
    missing = [c for c in expected_cols if c not in cols]
    if missing:
        raise ValueError(
            f"[{dataset_name}] missing columns:\n"
            f"{missing}\n\n"
            f"Available columns:\n{cols}"
        )
    logging.info(f" - [{dataset_name}] column check passed")
    minorsep()
    return True

# Read CSV safely using automatic encoding fallback
def safe_read_csv(path_obj, label, **kwargs):
    if "low_memory" not in kwargs:
        kwargs["low_memory"] = False
    check_file(path_obj, label)
    last_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(path_obj, encoding=enc, **kwargs)
            logging.info(
                f" - Loaded {label} "
                f"(encoding={enc}) | "
                f"rows={len(df):,} cols={len(df.columns)}"
            )
            return df
        except UnicodeDecodeError as e:
            last_error = e
    raise RuntimeError(
        f" - Failed loading {label}. "
        f"None of the tested encodings worked.\n"
        f"Last error:\n{last_error}"
    )

# Read Excel safely      
def safe_read_excel(path_obj, label, **kwargs):
    check_file(path_obj, label)
    try:
        df = pd.read_excel(path_obj, **kwargs)
        logging.info(f" - Loaded {label} | rows={len(df)} cols={len(df.columns)}")
        return df
    except Exception as e:
        raise RuntimeError(f" - Failed loading {label}: {e}")

# Strip leading/trailing whitespace from selected string columns
def strip_whitespace_columns(df, columns, dataset_name="dataset"):
    existing_cols = [c for c in columns if c in df.columns]
    if not existing_cols:
        logging.info(f" - No whitespace-cleanup columns found in {dataset_name}")
        return df
    for col in existing_cols:
        # only process non-null values
        df[col] = df[col].apply(
            lambda x: x.strip() if isinstance(x, str) else x
        )
    logging.info(
        f" - Whitespace cleanup applied to {dataset_name} | "
        f"Columns: {', '.join(existing_cols)}"
    )
    return df


# Seed
GLOBAL_SEED = 1
np.random.seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

majorsep()
logging.info(f"Imports & logging setup done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Check if necessary columns exist in raw input data files
majorsep()
logging.info(f"Checking if necessary columns exist in raw input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Data folder
DATA_DIR = Path("../data") # specify path to your working folder (default: 'data' folder of the repository)
if not DATA_DIR.exists(): raise FileNotFoundError(f"Working folder not found: {DATA_DIR.resolve()} → specify DATA_DIR\n")

# Lookup folder
LOOKUP_DIR = Path(DATA_DIR / "lookups")
LOOKUP_DIR.mkdir(exist_ok=True)


# Specify paths to the source databases and sample dataset
# CPDat
check_cols_before_load(DATA_DIR / "CPDat" / "product_compositions.tsv", ["cpdat_product_id", "inchikey"], "CPDat product compositions")
check_cols_before_load(DATA_DIR / "CPDat" / "products.tsv", ["cpdat_product_id", "product_title"], "CPDat products metadata")
check_cols_before_load(DATA_DIR / "CPDat" / "product_use_categories.tsv", ["cpdat_product_id", "puc_kind", "puc_general_category", "puc_harmonized"], "CPDat PUC")


# Household products
check_cols_before_load(DATA_DIR / "Own_Household_Products.csv", ["Brand-free Name", "Feature_Name", "InChIKey"], "Household products")
check_cols_before_load(DATA_DIR / "Own_Household_Products_Metadata.csv", ["MySysID", "Group1", "Group2", "Group3"], "Household products metadata")


# Consumer products - Literature
check_cols_before_load(DATA_DIR / "Literature_Products.csv", ["puc_harmonized", "Brand-free Name", "MySysID", "Feature_Name", "InChIKey", "Sample Concentration"], "Consumer products from literature")


# FooDB
check_cols_before_load(DATA_DIR / "FooDB.csv",
                       ['cont_id\tfood_name_cont\torig_food_id\torig_food_scientific_name\torig_food_part\torig_source_id\torig_feature_name_cont\torig_content\torig_min\t'
                       'orig_max\torig_unit\tfood_id\tfood_sub_id\tfood_name_food\torig_food_common_name\tfood_name_scientific\tdescription\titis_id\tlegacy_id\tfood_group\t'
                       'food_subgroup\tfood_type\tfood_group_harmonized\tncbi_taxonomy_id\tpublic_food_id\tcpd_id\tpublic_cpd_id\tinchikey\torig_feature_name_cpd\t'
                       'feature_name_pubchem\tiupac_name\tiupac_name_pubchem\tcpd_description\tcas\tsmiles\tinchi\tmono_mass\tsuperclass\tclass_classyfire\tsubclass'], 
                       "FooDB")


# SPECIEUROPE
check_cols_before_load(DATA_DIR / "SPECIEUROPE.xlsx", ["PROFILE_CODE", "PROFILE_NAME", "Feature name", "InChIKey"], "SPECIEUROPE")


# SPECIATE
check_cols_before_load(DATA_DIR / "SPECIATE.xlsx",
    ["PROFILE_CODE", "PROFILE_NAME", "Feature name", "InChIKey", "CATEGORY_LEVEL_1_Generation_Mechanism", "CATEGORY_LEVEL_2_Sector_Equipment", "CATEGORY_LEVEL_3_Fuel_Product"],
    "SPECIATE")


# Samples
check_cols_before_load(DATA_DIR / "Sample_Data.xlsx", ["inchikey", "Concentration Type", "Feature Name", "Class"], "Sample dataset")


logging.info(f"Done checking if necessary columns exist in raw input data files | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Load raw input data
majorsep()
logging.info(f"Loading raw input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# CPDat
products_0 = safe_read_csv(DATA_DIR / "CPDat" / "product_compositions.tsv", "CPDat product compositions", sep="\t")
products_meta_0 = safe_read_csv(DATA_DIR / "CPDat" / "products.tsv", "CPDat products metadata", sep="\t")
puc_0 = safe_read_csv(DATA_DIR / "CPDat" / "product_use_categories.tsv", "CPDat PUC", sep="\t")

logging.info(f" - Loaded CPDat | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Household products
household_products_0 = safe_read_csv(DATA_DIR / "Own_Household_Products.csv", "Household products", sep=",")
household_meta_0 = safe_read_csv(DATA_DIR / "Own_Household_Products_Metadata.csv", "Household products metadata", sep=",")

logging.info(f" - Loaded household products data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Consumer products - Literature
cons_prods_0 = safe_read_csv(DATA_DIR / "Literature_Products.csv", "Consumer products from literature", sep=",")

logging.info(f" - Loaded consumer products from literature | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# FooDB
foodb_0 = safe_read_csv(DATA_DIR / "FooDB.csv", "FooDB", sep="\t") # tab-separated despite the .csv extension

logging.info(f" - Loaded FooDB | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# SPECIEUROPE
specieurope_0 = safe_read_excel(DATA_DIR / "SPECIEUROPE.xlsx", "SPECIEUROPE", sheet_name=0, engine="openpyxl")

logging.info(f" - Loaded SPECIEUROPE | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# SPECIATE
speciate_0 = safe_read_excel(DATA_DIR / "SPECIATE.xlsx", "SPECIATE", sheet_name=0, engine="openpyxl")

# Remove aggregate/non-specific SPECIATE profiles
speciate_0 = speciate_0[speciate_0["PROFILE_NAME"] != "Overall Average"].copy()
logging.info(f" - Removed 'Overall Average' SPECIATE rows | Remaining rows: {len(speciate_0)}")

logging.info(f" - Loaded SPECIATE & dropped 'Overall Average' source | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Samples
dust_0_full = safe_read_excel(DATA_DIR / "Sample_Data.xlsx", "Sample dataset", sheet_name=0, engine="openpyxl")

logging.info(f" - Loaded sample data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


logging.info(f"Done loading raw input data | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Whitespace cleanup
majorsep()
logging.info(f"Running whitespace cleanup for selected columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Strip leading/trailing spaces from column names (e.g., "InChIKey " → "InChIKey")
# - Done first, so that next step (values) also finds columns whose names had spaces (e.g., "InChIKey ")
# - Stops with an error if stripping would create two columns with the same name (e.g., "Name" and "Name ")
def strip_column_names(df, dataset_name="dataset"):
    old_names = list(df.columns)
    new_names = [c.strip() if isinstance(c, str) else c for c in old_names]
    changed = [o for o, n in zip(old_names, new_names) if o != n]

    if not changed:
        logging.info(f" - Column names already clean in {dataset_name}")
        return df

    duplicates = sorted({n for n, k in Counter(new_names).items() if k > 1}, key=str)
    if duplicates:
        raise ValueError(f"Stripping column names in {dataset_name} would create duplicate columns: {duplicates}")

    df.columns = new_names
    logging.info(f" - Column names stripped in {dataset_name}: {len(changed):,} | e.g. {[repr(c) for c in changed[:5]]}")
    return df

logging.info(f"Stripping whitespace from column names")
for df_raw, df_label in [
    (products_0, "CPDat product compositions"),
    (products_meta_0, "CPDat products metadata"),
    (puc_0, "CPDat PUC"),
    (household_products_0, "Household products"),
    (household_meta_0, "Household products metadata"),
    (cons_prods_0, "Consumer products from literature"),
    (foodb_0, "FooDB"),
    (specieurope_0, "SPECIEUROPE"),
    (speciate_0, "SPECIATE"),
    (dust_0_full, "Sample dataset"),
]:
    strip_column_names(df_raw, df_label)
minorsep()


# Strip leading/trailing spaces from VALUES in selected columns
logging.info(f"Stripping whitespace from values in selected columns")

# Columns requiring whitespace cleanup
STRIP_COLUMNS = [
    "cpdat_product_id", "product_title", "puc_kind", "puc_general_category", "puc_harmonized",
    "product_id", "product_name", "PROFILE_CODE", "PROFILE_NAME", "Feature name",
    "CATEGORY_LEVEL_1_Generation_Mechanism", "CATEGORY_LEVEL_2_Sector_Equipment", "CATEGORY_LEVEL_3_Fuel_Product",
    "Category", "Group1", "Group2", "Group3", "Brand-free Name", "Feature_Name",
    "Feature Name", "Concentration Type", "food_group", "food_subgroup", "food_group_harmonized",
    "food_name_food", "orig_food_common_name", "orig_feature_name_cpd", "feature_name_pubchem",
    "inchikey", "InChIKey", "InChiKey"
]

# CPDat
products_strip = strip_whitespace_columns(products_0, STRIP_COLUMNS, "CPDat product compositions")
products_meta_strip = strip_whitespace_columns(products_meta_0, STRIP_COLUMNS, "CPDat products metadata")
puc_strip = strip_whitespace_columns(puc_0, STRIP_COLUMNS, "CPDat PUC")

# Household products
household_products_strip = strip_whitespace_columns(household_products_0, STRIP_COLUMNS, "Household products")
household_meta_strip = strip_whitespace_columns(household_meta_0, STRIP_COLUMNS, "Household products metadata")

# Consumer products from literature
cons_prods_strip = strip_whitespace_columns(cons_prods_0, STRIP_COLUMNS, "Consumer products from literature")

# FooDB
foodb_strip = strip_whitespace_columns(foodb_0, STRIP_COLUMNS, "FooDB")

# Specieurope
specieurope_strip = strip_whitespace_columns(specieurope_0, STRIP_COLUMNS, "SPECIEUROPE")

# SPECIATE
speciate_strip = strip_whitespace_columns(speciate_0, STRIP_COLUMNS, "SPECIATE")

# Samples
dust_full_strip = strip_whitespace_columns(dust_0_full, STRIP_COLUMNS, "Sample dataset")

minorsep()
logging.info(f"Done running whitespace cleanup for selected columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# InChIKey cleanup (all source DBs & dust)
# - works on the *_strip DFs, so every later step (suffixes, standardization, aggregation, masking, dust samples,
#   all saved files) uses the corrected InChIKeys
# - obvious junk (empty, "nan", "0", numbers, values without any letter) → set to missing (rows are later dropped
#   exactly like rows without an InChIKey)
# - digit zero '0' → letter 'O' (real InChIKeys contain only letters A-Z and hyphens, never digits)
# - upper case; odd-looking but plausible keys are kept and only listed in the log
majorsep()
logging.info(f"Cleaning InChIKeys in all source DBs & dust | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

INCHIKEY_DFS = { # DF name → label for the log (the InChIKey column is found automatically)
    "products_strip": "CPDat product compositions",
    "household_products_strip": "Household products",
    "cons_prods_strip": "Consumer products from literature",
    "foodb_strip": "FooDB",
    "specieurope_strip": "SPECIEUROPE",
    "speciate_strip": "SPECIATE",
    "dust_strip": "Dust dataset",
    "dust_full_strip": "Full dust dataset",
}
INCHIKEY_COL_NAMES = ["inchikey", "InChIKey", "InChiKey"] # possible names of the InChIKey column
INCHIKEY_MISSING = {"", "nan", "none", "na", "n/a", "null", "<na>", "-", "--"}
INCHIKEY_STANDARD = r"[A-Z]{14}-[A-Z]{10}-[A-Z]" # standard format; only used to report odd keys (they are kept)

def clean_inchikeys(s):
    """Returns (cleaned Series, n_junk, list of corrected keys, list of odd keys)."""
    st = s.astype("string").str.strip().str.upper()
    st = st.mask(st.str.lower().isin(INCHIKEY_MISSING))                   # placeholders → missing
    is_number = pd.to_numeric(st, errors="coerce").notna()                # "0", "12.5", "1E3", ...
    no_letter = ~st.str.contains(r"[A-Z]", regex=True).fillna(False)      # e.g. "10-20"
    junk = st.notna() & (is_number | no_letter)
    st = st.mask(junk)                                                    # junk → missing
    fixed = st.str.replace("0", "O", regex=False)                         # digit 0 → letter O
    zero_changed = st.notna() & (fixed != st)
    odd = fixed.notna() & ~fixed.str.fullmatch(INCHIKEY_STANDARD).fillna(False)
    out = fixed.astype(object).where(fixed.notna(), np.nan)               # back to normal object column with NaN
    return out, int(junk.sum()), st[zero_changed].unique().tolist(), fixed[odd].unique().tolist()

for df_name, label in INCHIKEY_DFS.items():
    if df_name not in globals():
        logging.warning(f" - {label}: DF '{df_name}' not found → skipped")
        minorsep()
        continue
    df = globals()[df_name]
    ik_cols = [c for c in INCHIKEY_COL_NAMES if c in df.columns]
    if not ik_cols:
        logging.warning(f" - {label}: no InChIKey column found → skipped")
        minorsep()
        continue

    for col in ik_cols:
        df[col], n_junk, zero_fixed, odd_keys = clean_inchikeys(df[col])
        msg = f" - {label} [{col}]: junk → missing: {n_junk:,} rows | '0' → 'O' corrected: {len(zero_fixed):,} keys"
        msg += f" | non-standard keys kept: {len(odd_keys):,}"
        logging.info(msg)

        if zero_fixed:
            logging.info(f"   - corrected (before): {zero_fixed[:5]}")
        if odd_keys:
            logging.info(f"   - non-standard (kept): {odd_keys[:5]}")
        minorsep()
        
logging.info(f"Done cleaning InChIKeys | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create copies of raw dfs
majorsep()
logging.info(f"Creating working copies of raw DFs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

products = products_strip.copy()
products_meta = products_meta_strip.copy()
puc = puc_strip.copy()
household_products = household_products_strip.copy()
household_meta = household_meta_strip.copy()
cons_prods = cons_prods_strip.copy()
foodb = foodb_strip.copy()
specieurope = specieurope_strip.copy()
speciate = speciate_strip.copy()
dust_full = dust_full_strip.copy()

logging.info(f"Created working copies of raw DFs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Add suffixes to IDs to distinguish between various databases
majorsep()
logging.info(f"Adding suffixes to IDs to distinguish between various databases | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Create copies of DFs
products_suf = products.copy()
products_meta_suf = products_meta.copy()
puc_suf = puc.copy()
household_products_suf = household_products.copy()
household_meta_suf = household_meta.copy()
cons_prods_suf = cons_prods.copy()
foodb_suf = foodb.copy()
specieurope_suf = specieurope.copy()
speciate_suf = speciate.copy()

# Add suffix "_CPD" to cpdat_product_id
products_suf["cpdat_product_id"] = (products_suf["cpdat_product_id"].astype("Int64").astype(str) + "_CPD")
products_meta_suf["cpdat_product_id"] = (products_meta_suf["cpdat_product_id"].astype("Int64").astype(str) + "_CPD")
puc_suf["cpdat_product_id"] = (puc_suf["cpdat_product_id"].astype("Int64").astype(str) + "_CPD")
logging.info(f" - Added suffixes to CPDat IDs")


# Add suffix "_HHP" to MySysID
household_products_suf["MySysID"] = (household_products_suf["MySysID"].astype(str) + "_HHP")
household_meta_suf["MySysID"] = (household_meta_suf["MySysID"].astype(str) + "_HHP")
logging.info(f" - Added suffixes to household product & metadata IDs")


# Add suffix "_COP" to MySysID
cons_prods_suf["MySysID"] = (cons_prods_suf["MySysID"].astype(str) + "_COP")
logging.info(f" - Added suffixes to consumer product IDs")


# Add suffix "_FDB" to food_sub_id
foodb_suf["food_sub_id"] = (foodb_suf["food_sub_id"].astype(str) + "_FDB")
logging.info(f" - Added suffixes to FooDB IDs")


# Add suffix "_SEU" to PROFILE_CODE
specieurope_suf["PROFILE_CODE"] = (specieurope_suf["PROFILE_CODE"].astype("Int64").astype(str) + "_SEU")
logging.info(f" - Added suffixes to SPECIEUROPE IDs")


# Add suffix "_SUS" to PROFILE_CODE
speciate_suf["PROFILE_CODE"] = (speciate_suf["PROFILE_CODE"].astype(str) + "_SUS")
logging.info(f" - Added suffixes to SPECIATE IDs")

minorsep()
logging.info(f"Added suffixes to IDs to distinguish between various databases | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save DBs with suffixes in IDs
majorsep()
logging.info(f"Saving all DBs with suffixes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

SUFFIX_DIR = Path(DATA_DIR / "databases_id_w_suffix")
SUFFIX_DIR.mkdir(exist_ok=True)

products_suf.to_csv(SUFFIX_DIR / "cpdat_products_suffix.csv", index=False)
products_meta_suf.to_csv(SUFFIX_DIR / "cpdat_products_meta_suffix.csv", index=False)
puc_suf.to_csv(SUFFIX_DIR / "cpdat_puc_suffix.csv", index=False)
household_meta_suf.to_csv(SUFFIX_DIR / "household_metadata_suffix.csv", index=False)
household_products_suf.to_csv(SUFFIX_DIR / "household_products_no-suffix.csv", index=False)
cons_prods_suf.to_csv(SUFFIX_DIR / "consumer_products_suffix.csv", index=False)
foodb_suf.to_csv(SUFFIX_DIR / "foodb_suffix.csv", index=False)
specieurope_suf.to_csv(SUFFIX_DIR / "specieurope_suffix.csv", index=False)
speciate_suf.to_csv(SUFFIX_DIR / "speciate_suffix.csv", index=False)

logging.info(f"Saved all DBs with suffixes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Standardize and harmonize source databases
majorsep()
logging.info(f"Standardizing contaminant source databases | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Create copies of DFs
products_std = products_suf.copy()
products_meta_std = products_meta_suf.copy()
puc_std = puc_suf.copy()

household_products_std = household_products_suf.copy()
household_meta_std = household_meta_suf.copy()

cons_prods_std = cons_prods_suf.copy()
foodb_std = foodb_suf.copy()
specieurope_std = specieurope_suf.copy()
speciate_std = speciate_suf.copy()


# Keep relevant CPDat PUC cols
puc_std = puc_std[[
    "cpdat_product_id",
    "puc_kind",
    "puc_general_category",
    "puc_harmonized"
]].drop_duplicates()

puc_std = (puc_std.sort_values("cpdat_product_id").drop_duplicates(subset="cpdat_product_id", keep="first"))

# Standardize PUC
puc_std = puc_std.rename(columns={
    "cpdat_product_id": "source_id",
    "puc_kind": "source_kind",
    "puc_general_category": "source_general_category",
    "puc_harmonized": "source_harmonized"
})

logging.info(f" - Unique PUC products: {puc_std['source_id'].nunique():,}")


# Must ensure 'inchikey' column exists here (merge if needed from original CPDat)
if "inchikey" not in products_std.columns:
    raise ValueError("inchikey column missing in product_compositions")
    
# CPDat simplified
products_simple = products_std[["cpdat_product_id", "inchikey"]].dropna().drop_duplicates()

logging.info(f" - CPDat simplified rows: {len(products_simple):,}")


# Standardized source schema
SOURCE_COLUMNS = [
    "source_id",
    "source_name",
    "inchikey",
    "source_origin",
    "source_kind",
    "source_general_category",
    "source_harmonized"
]


# CPDat
cpdat_sources = (
    products_simple
    .rename(columns={
        "cpdat_product_id": "source_id",
        "puc_kind": "source_kind",
        "puc_general_category": "source_general_category",
        "puc_harmonized": "source_harmonized"
    })
)

# Add product titles from products_meta
cpdat_titles = products_meta_std[[
    "cpdat_product_id",
    "product_title"
]].rename(columns={
    "cpdat_product_id": "source_id",
    "product_title": "source_name",
    "puc_kind": "source_kind",
    "puc_general_category": "source_general_category",
    "puc_harmonized": "source_harmonized"
})

# Merge titles into CPDat sources
cpdat_sources = cpdat_sources.merge(
    cpdat_titles,
    on="source_id",
    how="left"
)

cpdat_sources["source_origin"] = "CPDAT"

cpdat_sources = cpdat_sources.merge(
    puc_std.rename(columns={"cpdat_product_id": "source_id"}),
    on="source_id",
    how="left"
)

# Standardize products_meta
products_meta_std = products_meta_std.rename(columns={
    "cpdat_product_id": "source_id",
    "product_title": "source_name",
    "puc_kind": "source_kind",
    "puc_general_category": "source_general_category",
    "puc_harmonized": "source_harmonized"
})

logging.info(f" - Standardized CPDat & CPDat metadata")
minorsep()


# Household products
household_products_std = household_products_std.rename(columns={
    "MySysID": "source_id",
    "Brand-free Name": "source_name"
})

household_products_std["source_origin"] = "HOUSEHOLD_PRODUCTS"

# for col in ["puc_kind", "puc_general_category", "puc_harmonized"]:
for col in ["source_kind", "source_general_category", "puc_harmonized"]:
    if col not in household_products_std.columns:
        household_products_std[col] = np.nan

# Map household products ontology columns into PUC-like structure
household_products_std = household_products_std.rename(columns={
    "Feature_Name": "feature_name",
    "InChIKey": "inchikey",
    "Group3": "source_type",
    "puc_harmonized": "source_harmonized"
})

logging.info(f" - Mapped household products ontology columns into PUC-like metadata")

logging.info(f" - Standardized household products")
minorsep()


# Consumer products from literature
cons_prods_std = cons_prods_std.rename(columns={
    "MySysID": "source_id",
    "Brand-free Name": "source_name",
    "Feature_Name": "feature_name",
    "InChIKey": "inchikey"
})

cons_prods_std["source_origin"] = "LIT_CONSUMER_PRODUCTS"

# Map lit. consumer products ontology columns into PUC-like structure
cons_prods_std = cons_prods_std.rename(columns={
    "Group1": "source_kind",
    "Group2": "source_general_category",
    "Group3": "source_type",
    "puc_harmonized": "source_harmonized"
})

logging.info(f" - Mapped lit. consumer products ontology columns into PUC-like metadata")
logging.info(f" - Standardized lit. consumer products")
minorsep()


# FooDB
foodb_std = foodb_std.rename(columns={
    "food_sub_id": "source_id",
    "orig_food_common_name": "source_name",
    "orig_feature_name_cpd": "feature_name"
})

foodb_std["source_origin"] = "FOODB"

# Map FooDB ontology columns into PUC-like structure
foodb_std = foodb_std.rename(columns={
    "food_group": "source_kind",
    "food_subgroup": "source_general_category",
    "food_type": "source_type",
    "food_group_harmonized": "source_harmonized"
})

logging.info(f" - Mapped FooDB ontology columns into PUC-like metadata")


# Delete semi-duplicates (same inchikey), like 'Niacin' vs. 'Niacin, from tryptophan'
# Keep only highest orig_content per source_id + inchikey, i.e. 'Niacin, total'
required_cols = ['source_id', 'inchikey', 'orig_content']

if all(col in foodb_std.columns for col in required_cols):
    rows_before = len(foodb_std)

    # convert to numeric safely
    foodb_std['orig_content_num'] = pd.to_numeric(foodb_std['orig_content'], errors='coerce')

    # index of highest value within each source_id + inchikey group
    idx_keep = (foodb_std.groupby(['source_id', 'inchikey'])['orig_content_num'].idxmax())
    foodb_std = foodb_std.loc[idx_keep].copy()

    # remove helper column
    foodb_std.drop(columns=['orig_content_num'], inplace=True)
    rows_after = len(foodb_std)
else:
    logging.info(f"Required FooDB columns missing\n")
    
logging.info(f" - Deleted {rows_before - rows_after:,} semi-duplicates (same inchikey) from FooDB")
logging.info(f" - Standardized FooDB")
minorsep()


# SPECIEUROPE
specieurope_std = specieurope_std.rename(columns={
    "PROFILE_CODE": "source_id",
    "PROFILE_NAME": "source_name",
    "Feature name": "feature_name",
    "InChIKey": "inchikey"
})

specieurope_std["source_origin"] = "SPECIEUROPE"

# Map SPECIEUROPE ontology columns into PUC-like structure
specieurope_std = specieurope_std.rename(columns={
    "CATEGORY_LEVEL_1_Generation_Mechanism": "source_kind",
    "CATEGORY_LEVEL_2_Sector_Equipment": "source_general_category",
    "CATEGORY_LEVEL_3_Fuel_Product": "source_type"
})

logging.info(f" - Mapped SPECIEUROPE ontology columns into PUC-like metadata")

# Build harmonized SPECIEUROPE ontology string
specieurope_std["source_harmonized"] = (
    specieurope_std["source_kind"].fillna("Unknown").astype(str)
    + " | " +
    specieurope_std["source_general_category"].fillna("Unknown").astype(str)
    + " | " +
    specieurope_std["source_type"].fillna("Unknown").astype(str)
)

logging.info(f" - Constructed SPECIEUROPE source_harmonized field")

for col in ["source_kind", "source_general_category", "source_harmonized"]:
    if col not in specieurope_std.columns:
        specieurope_std[col] = np.nan

logging.info(f" - Standardized SPECIEUROPE")
minorsep()


# SPECIATE
speciate_std = speciate_std.rename(columns={
    "PROFILE_CODE": "source_id",
    "PROFILE_NAME": "source_name",
    "Feature name": "feature_name",
    "InChIKey": "inchikey"
})

speciate_std["source_origin"] = "SPECIATE"

# Map SPECIATE ontology columns into PUC-like structure
speciate_std = speciate_std.rename(columns={
    "CATEGORY_LEVEL_1_Generation_Mechanism": "source_kind",
    "CATEGORY_LEVEL_2_Sector_Equipment": "source_general_category",
    "CATEGORY_LEVEL_3_Fuel_Product": "source_type"
})

logging.info(f" - Mapped SPECIATE ontology columns into PUC-like metadata")

# Build harmonized SPECIATE ontology string
speciate_std["source_harmonized"] = (
    speciate_std["source_kind"].fillna("Unknown").astype(str)
    + " | " +
    speciate_std["source_general_category"].fillna("Unknown").astype(str)
    + " | " +
    speciate_std["source_type"].fillna("Unknown").astype(str)
)

logging.info(f" - Constructed SPECIATE source_harmonized field")

for col in ["source_kind", "source_general_category", "source_harmonized"]:
    if col not in speciate_std.columns:
        speciate_std[col] = np.nan

logging.info(f" - Standardized SPECIATE")
minorsep()


# Final source concatenation (concatenate all sources)
all_sources_0 = pd.concat([
    household_products_std[SOURCE_COLUMNS],
    cons_prods_std[SOURCE_COLUMNS],
    foodb_std[SOURCE_COLUMNS],
    specieurope_std[SOURCE_COLUMNS],
    speciate_std[SOURCE_COLUMNS],
    cpdat_sources[SOURCE_COLUMNS]
], ignore_index=True)

logging.info(f" - Unified source rows: {len(all_sources_0):,}")
logging.info(f" - Unique source IDs: {all_sources_0['source_id'].nunique():,}")

# Remove invalid rows
all_sources_0 = all_sources_0.dropna(subset=["inchikey"])
logging.info(f" - Rows after inchikey filtering: {len(all_sources_0):,}")

minorsep()
logging.info(f"Done standardizing source databases | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Add the two-level reporting categories source_class & source_class_domain from the mapping file
# - source_class_mapping.xlsx, sheet "source_class_mapping"
# - one row per source_origin × source_kind × source_general_category × source_harmonized)
# - they are reporting labels only: priors, aggregation & the set cover do NOT use them
# - the key is cleaned exactly as in the export that produced the mapping (missing → "Unknown", leading/trailing spaces removed), so both sides match
majorsep()
logging.info(f"Adding source_class & source_class_domain & dropping columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f"Adding source_class & source_class_domain to all_sources_0 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Path to the mapping file
SOURCE_CLASS_FILE = LOOKUP_DIR / "source_class_mapping.xlsx"
SOURCE_CLASS_SHEET = "source_class_mapping"
CLASS_KEY_COLS = ["source_origin", "source_kind", "source_general_category", "source_harmonized"]
MISSING_LABELS = {"", "nan", "none", "na", "n/a", "null", "<na>"} # text that pandas reads as missing

# Check if necessary columns exist
check_cols_before_load(SOURCE_CLASS_FILE, ["source_origin", "source_kind", "source_general_category", "source_harmonized", "source_class", "source_class_domain"], "source_class mapping")

def clean_class_key(series):
    s = series.astype("string").str.strip()
    s = s.where(~s.isna() & ~s.str.lower().isin(MISSING_LABELS), "Unknown")
    return s.fillna("Unknown").astype(str)

# Read file to DF
source_class_map = pd.read_excel(SOURCE_CLASS_FILE, sheet_name=SOURCE_CLASS_SHEET, dtype=str, keep_default_na=False, engine="openpyxl")

for col in CLASS_KEY_COLS:
    source_class_map[col] = clean_class_key(source_class_map[col])

source_class_map = (source_class_map[CLASS_KEY_COLS + ["source_class", "source_class_domain"]].drop_duplicates())

conflicts = source_class_map[source_class_map.duplicated(subset=CLASS_KEY_COLS, keep=False)]
if not conflicts.empty:
    raise ValueError(f" - source_class mapping has {len(conflicts):,} rows with the same key but different classes, e.g.:\n   "
                     f"{conflicts.head(10)}")

# Temporary cleaned key columns on the source table (the original columns stay unchanged)
key_tmp = [f"_key_{c}" for c in CLASS_KEY_COLS]
for col, tmp in zip(CLASS_KEY_COLS, key_tmp):
    all_sources_0[tmp] = clean_class_key(all_sources_0[col])

rows_before_class = len(all_sources_0)
all_sources_0 = all_sources_0.merge(
    source_class_map.rename(columns=dict(zip(CLASS_KEY_COLS, key_tmp))),
    on=key_tmp,
    how="left",
    validate="many_to_one"
)
assert len(all_sources_0) == rows_before_class, "source_class merge changed the number of rows"

# Sources whose categorization is not in the mapping file (e.g., new data) → logged, saved, labelled
unmapped = all_sources_0["source_class"].isna()
if unmapped.any():
    unmapped_keys = (all_sources_0.loc[unmapped, key_tmp].drop_duplicates()
        .rename(columns=dict(zip(key_tmp, CLASS_KEY_COLS))))
    logging.warning(f" - Sources NOT in the source_class mapping: {all_sources_0.loc[unmapped, 'source_id'].nunique():,} "
                    f"({len(unmapped_keys):,} categorizations) → labelled 'Unmapped Sources | <origin>'")
    unmapped_keys.to_csv(LOOKUP_DIR / "source_class_unmapped.csv", index=False)
    logging.warning(f" - List saved to lookups/source_class_unmapped.csv → add these rows to the mapping file and re-run")
    all_sources_0.loc[unmapped, "source_class"] = "Unmapped Sources | " + all_sources_0.loc[unmapped, "source_origin"].astype(str)
    all_sources_0.loc[unmapped, "source_class_domain"] = "Unmapped Sources"
else:
    logging.info(f" - all_sources_0 mapped to a source_class")

all_sources_0 = all_sources_0.drop(columns=key_tmp)

logging.info(f" - Unique source_class: {all_sources_0['source_class'].nunique():,} | "
             f"unique source_class_domain: {all_sources_0['source_class_domain'].nunique():,}")
logging.info(f"Done adding source_class & source_class_domain to all_sources_0 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()


# Add source_class & source_class_domain also to the separate standardized DBs (..._std)
# and clean them before they are saved in the "# Save standardized DBs" cell:
# - add the two class columns (same mapping file & same key cleaning as for all_sources_0 above)
# - drop the uninformative 'ID' and '#' columns (if they exist)
# - optionally drop all totally empty columns (switch DROP_EMPTY_COLS_STD)
# - all_sources_0 & sources_meta are NOT changed by this section (later cells depend on them)
logging.info(f"Adding source_class & source_class_domain & cleaning separate standardized DBs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# DROP_EMPTY_COLS_STD = True # True = drop totally empty columns from the standardized DBs; False = keep them
DROP_EMPTY_COLS_STD = False
# Columns to drop from the standardized DBs (if they exist)
DROP_COLS_STD = ["ID", "#"]
# Columns that are never dropped, even if empty
PROTECTED_COLS_STD = set(SOURCE_COLUMNS + ["feature_name", "source_type", "cpdat_product_id", "source_class", "source_class_domain"])
# Collects unmapped categorizations from the standardized DBs
unmapped_std_list = []

def add_source_class(df, db_name, origin=None):
    """Add source_class & source_class_domain to one standardized DB
    origin = value used for source_origin if the DB has no source_origin column (e.g., "CPDAT" for puc_std)"""
    df = df.copy()
    missing_keys = [c for c in CLASS_KEY_COLS if c not in df.columns and not (c == "source_origin" and origin is not None)]
    if missing_keys:
        logging.info(f" - {db_name}: no source_class added (missing key columns: {missing_keys})")
        return df

    df = df.drop(columns=["source_class", "source_class_domain"], errors="ignore") # avoid duplicates on re-run

    # temporary cleaned key columns (the original columns stay unchanged)
    for col, tmp in zip(CLASS_KEY_COLS, key_tmp):
        if col in df.columns:
            df[tmp] = clean_class_key(df[col])
        else:
            df[tmp] = str(origin) # only source_origin can get here

    rows_before = len(df)
    df = df.merge(
        source_class_map.rename(columns=dict(zip(CLASS_KEY_COLS, key_tmp))),
        on=key_tmp,
        how="left",
        validate="many_to_one"
    )
    assert len(df) == rows_before, f"source_class merge changed the number of rows in {db_name}"

    # categorizations not in the mapping file → labelled 'Unmapped Sources | <origin>' (same as all_sources_0)
    unmapped_mask = df["source_class"].isna()
    if unmapped_mask.any():
        keys = (df.loc[unmapped_mask, key_tmp].drop_duplicates().rename(columns=dict(zip(key_tmp, CLASS_KEY_COLS))))
        keys.insert(0, "standardized_db", db_name)
        unmapped_std_list.append(keys)
        logging.warning(f" - {db_name}: {unmapped_mask.sum():,} rows ({len(keys):,} categorizations) NOT in the "
                        f"source_class mapping → labelled 'Unmapped Sources | <origin>'")
        df.loc[unmapped_mask, "source_class"] = "Unmapped Sources | " + df.loc[unmapped_mask, key_tmp[0]].astype(str)
        df.loc[unmapped_mask, "source_class_domain"] = "Unmapped Sources"

    df = df.drop(columns=key_tmp)
    logging.info(f" - {db_name}: source_class added | unique source_class: {df['source_class'].nunique():,}")
    return df

def clean_std_db(df, db_name):
    """Drop 'ID' & '#' columns and (if DROP_EMPTY_COLS_STD) all totally empty columns from one standardized DB."""
    # drop 'ID' and '#' (compared without surrounding spaces)
    drop_named = [c for c in df.columns if str(c).strip() in DROP_COLS_STD]

    # totally empty columns = only NaN and/or blank text
    drop_empty = []
    if DROP_EMPTY_COLS_STD:
        for c in df.columns:
            if c in drop_named or c in PROTECTED_COLS_STD:
                continue
            non_missing = df[c].dropna()
            if non_missing.empty or non_missing.astype(str).str.strip().eq("").all():
                drop_empty.append(c)

    if drop_named or drop_empty:
        df = df.drop(columns=drop_named + drop_empty)
    logging.info(f" - {db_name}: dropped {len(drop_named)} ID/# columns {drop_named} | "
                 f"{len(drop_empty)} empty columns {drop_empty}")
    return df

# Add source_class & source_class_domain (DBs without the key columns are skipped with a log message)
household_products_std = add_source_class(household_products_std, "household_products_std")
cons_prods_std = add_source_class(cons_prods_std, "cons_prods_std")
foodb_std = add_source_class(foodb_std, "foodb_std")
specieurope_std = add_source_class(specieurope_std, "specieurope_std")
speciate_std = add_source_class(speciate_std, "speciate_std")
cpdat_sources = add_source_class(cpdat_sources, "cpdat_sources")
puc_std = add_source_class(puc_std, "puc_std", origin="CPDAT")
products_meta_std = add_source_class(products_meta_std, "products_meta_std", origin="CPDAT")

# Save the unmapped categorizations of the standardized DBs (separate file; does not overwrite source_class_unmapped.csv)
if unmapped_std_list:
    pd.concat(unmapped_std_list, ignore_index=True).to_csv(LOOKUP_DIR / "source_class_unmapped_std_dbs.csv", index=False)
    logging.warning(f"List saved to lookups/source_class_unmapped_std_dbs.csv → add these rows to the mapping file and re-run")
    minorsep()
else:
    logging.info(f"All standardized DB rows mapped to a source_class | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    minorsep()

# Drop 'ID', '#' & (optionally) totally empty columns from all standardized DBs that are saved
logging.info(f"Dropping columns & cleaning separate standardized DBs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

products_std = clean_std_db(products_std, "products_std")
products_meta_std = clean_std_db(products_meta_std, "products_meta_std")
puc_std = clean_std_db(puc_std, "puc_std")
products_simple = clean_std_db(products_simple, "products_simple")
cpdat_sources = clean_std_db(cpdat_sources, "cpdat_sources")
cpdat_titles = clean_std_db(cpdat_titles, "cpdat_titles")
household_products_std = clean_std_db(household_products_std, "household_products_std")
cons_prods_std = clean_std_db(cons_prods_std, "cons_prods_std")
foodb_std = clean_std_db(foodb_std, "foodb_std")
specieurope_std = clean_std_db(specieurope_std, "specieurope_std")
speciate_std = clean_std_db(speciate_std, "speciate_std")

logging.info(f"Done dropping columns & cleaning separate standardized DBs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()
logging.info(f"Done adding source_class & source_class_domain & dropping columns | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Standardize source metadata
majorsep()
logging.info(f"Standardizing all source metadata | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Enforce unified metadata schema used by costfun
sources_meta = all_sources_0[[
    "source_id",
    "source_name",
    "source_origin",
    "source_kind",
    "source_general_category",
    "source_harmonized",
    "source_class",
    "source_class_domain"
]].copy()

# Normalize missing values
sources_meta["source_kind"] = sources_meta["source_kind"].fillna("Unknown")
sources_meta["source_general_category"] = sources_meta["source_general_category"].fillna("Unknown")
sources_meta["source_harmonized"] = sources_meta["source_harmonized"].fillna("Unknown")

logging.info(f" - Unified source metadata rows: {len(sources_meta):,}")
logging.info(f" - Unique source metadata IDs: {sources_meta['source_id'].nunique():,}")

logging.info(f"Done standardizing all source metadata | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save standardized DBs
majorsep()
logging.info(f"Saving all standardized source DBs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

STD_DIR = Path(DATA_DIR / "databases_standardized")
STD_DIR.mkdir(exist_ok=True)

products_std.to_csv(STD_DIR / "cpdat_products_standardized.csv", index=False)
products_meta_std.to_csv(STD_DIR / "cpdat_products_meta_standardized.csv", index=False)
puc_std.to_csv(STD_DIR / "cpdat_puc_standardized.csv", index=False)
products_simple.to_csv(STD_DIR / "cpdat_products_simple_standardized.csv", index=False)
cpdat_sources.to_csv(STD_DIR / "cpdat_sources_standardized.csv", index=False)
cpdat_titles.to_csv(STD_DIR / "cpdat_titles_standardized.csv", index=False)
logging.info(f" - Saved standardized CPDat DBs")

all_sources_0.to_csv(STD_DIR / "all_sources_standardized.csv", index=False)

sources_meta_unique_out = (sources_meta.sort_values("source_id", kind="stable")
    .drop_duplicates(subset="source_id", keep="first"))
sources_meta_unique_out.to_csv(STD_DIR / "sources_meta_standardized.csv", index=False)
logging.info(f" - sources_meta: {len(sources_meta):,} rows → {len(sources_meta_unique_out):,} unique sources saved")
# check: a source should have only one set of metadata
n_conflict = (sources_meta.drop_duplicates().groupby("source_id").size() > 1).sum()
if n_conflict:
    logging.warning(f" - {n_conflict:,} sources have more than one metadata row (first one kept)")
    
logging.info(f" - Saved standardized all sources DBs")

household_products_std.to_csv(STD_DIR / "household_products_standardized.csv", index=False)
logging.info(f" - Saved standardized household products DBs")

cons_prods_std.to_csv(STD_DIR / "lit_consumer_products_standardized.csv", index=False)
logging.info(f" - Saved standardized lit. consumer products DBs")

foodb_std.to_csv(STD_DIR / "foodb_standardized.csv", index=False)
logging.info(f" - Saved standardized FooDB")

specieurope_std.to_csv(STD_DIR / "specieurope_standardized.csv", index=False)
logging.info(f" - Saved standardized SPECIEUROPE")

speciate_std.to_csv(STD_DIR / "speciate_standardized.csv", index=False)
logging.info(f" - Saved standardized SPECIATE")

minorsep()
logging.info(f"Saved all standardized source DBs | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save all compounds (inchikey, names, CAS, DTXSID) across all source DBs
majorsep()
logging.info(f"Building all-compounds lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Separator for multiple values in one cell (not comma/tab → safe in CSV & Excel; split later with .str.split("; "))
MULTI_SEP = "; "

# Which input columns go where, per database
#  - only columns that exist in the DF are used (missing ones are logged and skipped)
#  - "main_name" = name used for 'feature_name' (and also added to 'all_feature_names')
#  - for SPECIATE & SPECIEUROPE the same column list is given, so it does not matter which file has which column
COMPOUND_COLS = {
    "HOUSEHOLD_PRODUCTS": {"df": "household_products_0", "inchikey": "InChIKey",
        "main_name": ["Feature_Name"], "other_names": ["Raw_Name", "PubChem_Name"],
        "cas": ["Raw_CAS"], "dtxsid": ["DTXSID"]},
    "LIT_CONSUMER_PRODUCTS": {"df": "cons_prods_0", "inchikey": "InChIKey",
        "main_name": ["Feature_Name"], "other_names": ["Raw_Name", "PubChem_Name"],
        "cas": ["Raw_CAS"], "dtxsid": ["DTXSID"]},
    "SPECIATE": {"df": "speciate_0", "inchikey": "InChIKey",
        "main_name": ["Feature name"], "other_names": ["Name_Original"],
        "cas": ["CAS#"], "dtxsid": ["DSSTox_ID"]},
    "SPECIEUROPE": {"df": "specieurope_0", "inchikey": "InChIKey",
        "main_name": ["Feature name"], "other_names": ["Name_Original"],
        "cas": ["CAS#"], "dtxsid": []},
    "FOODB": {"df": "foodb_0", "inchikey": "inchikey",
        "main_name": ["orig_feature_name_cpd"], "other_names": ["feature_name_pubchem", "iupac_name_pubchem"],
        "cas": ["cas"], "dtxsid": []},
    "CPDAT": {"df": "products_0", "inchikey": "inchikey",
        "main_name": ["curated_chemical_name"], "other_names": [],
        "cas": [], "dtxsid": []},
}

# Order in which databases are checked to pick 'feature_name' (first DB that has a name wins;
# within that DB the most frequent name is used). Reorder if you prefer another DB's names.
FEATURE_NAME_PRIORITY = ["CPDAT", "SPECIATE", "SPECIEUROPE", "FOODB", "LIT_CONSUMER_PRODUCTS", "HOUSEHOLD_PRODUCTS"]

MISSING_TEXT = {"", "nan", "none", "na", "n/a", "null", "<na>", "-", "--", "0", "0.0"} # "0" = empty placeholder
INCHIKEY_STANDARD = r"[A-Z]{14}-[A-Z]{10}-[A-Z]" # standard InChIKey format; only used to report odd keys (they are kept)
CAS_PATTERN = r"(\d{2,7}-\d{2}-\d)" # e.g., 117-81-7
DTXSID_PATTERN = r"(DTXSID\d+)" # e.g., DTXSID5020607

def clean_text(s):
    """Strip text and set placeholder text ('nan', 'NA', ...) to missing."""
    s = s.astype("string").str.strip()
    return s.mask(s.str.lower().isin(MISSING_TEXT))

def cas_from_digits(s):
    """CAS numbers stored without hyphens (e.g. 117817 → 117-81-7)."""
    s = s.str.replace(r"\.0$", "", regex=True) # numbers read from Excel as float, e.g. "1330207.0"
    digits = s.str.fullmatch(r"\d{5,10}").fillna(False)
    out = s.copy()
    out[digits] = s[digits].str[:-3] + "-" + s[digits].str[-3:-1] + "-" + s[digits].str[-1]
    return out

def is_junk_inchikey(s):
    """Obvious junk = numbers (e.g. "0", "12.5", "1e3") or values without any letter.
    Odd-looking keys (e.g. NIST23 keys like 'DRAWQKGUOR0SA-CLFAGFIQSA-N') are NOT junk and are kept."""
    is_number = pd.to_numeric(s, errors="coerce").notna()
    no_letter = ~s.str.contains(r"[A-Za-z]", regex=True).fillna(False)
    return s.notna() & (is_number | no_letter)

def fix_inchikey_zeros(s):
    """Real InChIKeys contain only letters A-Z and hyphens, never digits → digit zero '0' is a typo for letter 'O'.
    Apply only AFTER removing junk (otherwise e.g. '10-20' would turn into a letter-containing key)."""
    return s.str.replace("0", "O", regex=False)

def drop_float_zero(s):
    """Numbers read as float ("1333905.0") → "1333905"."""
    return s.str.replace(r"^(\d+)\.0$", r"\1", regex=True)

long_rows = [] # one row per inchikey × field × value

for db_rank, db in enumerate(FEATURE_NAME_PRIORITY):
    spec = COMPOUND_COLS[db]
    if spec["df"] not in globals():
        logging.warning(f" - {db}: DF '{spec['df']}' not found → skipped")
        continue
    df = globals()[spec["df"]]

    if spec["inchikey"] not in df.columns:
        logging.warning(f" - {db}: inchikey column '{spec['inchikey']}' missing → skipped")
        continue

    field_cols = {"main_name": spec["main_name"], "other_name": spec["other_names"],
                  "cas": spec["cas"], "dtxsid": spec["dtxsid"]}

    # Count invalid inchikeys once per DB; their rows are skipped below
    ik_all = clean_text(df[spec["inchikey"]]).str.upper()

    # Skip only obvious junk (numbers, no letters); keep odd-looking keys but report them
    ik_junk = is_junk_inchikey(ik_all)
    if ik_junk.any():
        logging.warning(f" - {db}: {ik_junk.sum():,} rows with a junk inchikey (e.g. {ik_all[ik_junk].unique()[:5].tolist()}) → skipped")

    # zero → letter O in the kept inchikeys (logged)
    ik_kept = ik_all[ik_all.notna() & ~ik_junk]
    ik_fixed = fix_inchikey_zeros(ik_kept)
    n_zero_fixed = ik_kept[ik_kept != ik_fixed].nunique()
    if n_zero_fixed:
        changed = ik_kept != ik_fixed
        examples = [f"{a} → {b}" for a, b in zip(ik_kept[changed].unique()[:3], ik_fixed[changed].unique()[:3])]
        logging.info(f" - {db}: {n_zero_fixed:,} inchikeys with digit '0' corrected to letter 'O' (e.g. {examples})")
    ik_odd = ~ik_fixed.str.fullmatch(INCHIKEY_STANDARD).fillna(False) # checked after the correction
    ik_all = ik_fixed # used for the log line below
    if ik_odd.any():
        logging.info(f" - {db}: {ik_all[ik_odd].nunique():,} non-standard inchikeys KEPT (e.g. {ik_all[ik_odd].unique()[:5].tolist()})")

    missing_cols = [c for cols in field_cols.values() for c in cols if c not in df.columns]
    if missing_cols:
        logging.warning(f" - {db}: columns not found (skipped): {missing_cols}")

    for field, cols in field_cols.items():
        for col in [c for c in cols if c in df.columns]:
            part = df[[spec["inchikey"], col]].copy()
            part.columns = ["inchikey", "value"]
            part["inchikey"] = clean_text(part["inchikey"]).str.upper()
            part["value"] = clean_text(part["value"])
            part = part.dropna(subset=["inchikey", "value"])
            part = part[~is_junk_inchikey(part["inchikey"])] # drop only junk inchikeys
            part["inchikey"] = fix_inchikey_zeros(part["inchikey"]) # digit '0' → letter 'O'
            part["value"] = drop_float_zero(part["value"])
            part = part.groupby(["inchikey", "value"], sort=False).size().reset_index(name="n") # n = rows with this value
            part["field"] = field
            part["db"] = db
            part["db_rank"] = db_rank
            long_rows.append(part)

    logging.info(f" - {db}: collected from '{spec['df']}'")
    minorsep()

compounds_long = pd.concat(long_rows, ignore_index=True)

# All inchikeys
all_compounds = pd.DataFrame({"inchikey": sorted(compounds_long["inchikey"].unique())})

# 2) feature_name: first DB in FEATURE_NAME_PRIORITY that has a main name; within it the most frequent name
main_names = compounds_long[compounds_long["field"] == "main_name"]
main_counts = (main_names.groupby(["inchikey", "db_rank", "value"], as_index=False)["n"].sum()
    .sort_values(["inchikey", "db_rank", "n", "value"], ascending=[True, True, False, True]))
feature_name = main_counts.drop_duplicates("inchikey")[["inchikey", "value"]].rename(columns={"value": "feature_name"})

def join_unique(values, case_insensitive=True):
    """Unique values in order of appearance, joined with MULTI_SEP."""
    seen, out = set(), []
    for v in values:
        key = v.lower() if case_insensitive else v
        if key not in seen:
            seen.add(key)
            out.append(v)
    return MULTI_SEP.join(out)

# 3) all_feature_names: main names first, then other names; DBs in FEATURE_NAME_PRIORITY order; case-insensitive unique
names = compounds_long[compounds_long["field"].isin(["main_name", "other_name"])].copy()
names["field_rank"] = (names["field"] == "other_name").astype(int)
names = names.sort_values(["inchikey", "field_rank", "db_rank"], kind="stable")
all_names = names.groupby("inchikey")["value"].agg(join_unique).reset_index(name="all_feature_names")

# 4) cas: extract all CAS-formatted numbers (cells may hold several, or digits without hyphens)
cas = compounds_long[compounds_long["field"] == "cas"].copy()
cas["value"] = cas_from_digits(cas["value"])
cas = cas.assign(value=cas["value"].str.findall(CAS_PATTERN)).explode("value").dropna(subset=["value"])
cas = cas.sort_values(["inchikey", "db_rank"], kind="stable")
cas_joined = cas.groupby("inchikey")["value"].agg(lambda v: join_unique(v, case_insensitive=False)).reset_index(name="cas")

# 5) dtxsid: extract all DTXSID IDs
dtx = compounds_long[compounds_long["field"] == "dtxsid"].copy()
dtx = dtx.assign(value=dtx["value"].str.upper().str.findall(DTXSID_PATTERN)).explode("value").dropna(subset=["value"])
dtx = dtx.sort_values(["inchikey", "db_rank"], kind="stable")
dtx_joined = dtx.groupby("inchikey")["value"].agg(lambda v: join_unique(v, case_insensitive=False)).reset_index(name="dtxsid")

# Combine
all_compounds = (all_compounds
    .merge(feature_name, on="inchikey", how="left", validate="one_to_one")
    .merge(all_names, on="inchikey", how="left", validate="one_to_one")
    .merge(cas_joined, on="inchikey", how="left", validate="one_to_one")
    .merge(dtx_joined, on="inchikey", how="left", validate="one_to_one"))

all_compounds = all_compounds[["inchikey", "feature_name", "all_feature_names", "cas", "dtxsid"]]

logging.info(f" - Unique compounds (inchikeys): {len(all_compounds):,}")
logging.info(f" - With feature_name: {all_compounds['feature_name'].notna().sum():,} | "
             f"with CAS: {all_compounds['cas'].notna().sum():,} | "
             f"with DTXSID: {all_compounds['dtxsid'].notna().sum():,}")
logging.info(f" - With >1 CAS: {all_compounds['cas'].str.contains(MULTI_SEP, regex=False).sum():,} | "
             f"with >1 DTXSID: {all_compounds['dtxsid'].str.contains(MULTI_SEP, regex=False).sum():,}")

# Save
all_compounds.to_csv(STD_DIR / "all_compounds_standardized.csv", index=False)
minorsep()
logging.info(f"Saved all_compounds_standardized.csv to: \n{STD_DIR}")

minorsep()
logging.info(f"Done building all-compounds lookup | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Aggregate equivalent products/sources
majorsep()
logging.info(f"Aggregating equivalent products/sources (same compound sets) per source_origin & source category | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

all_sources = all_sources_0.copy()

# Time stamp
aggregation_start = time.time()

# Use full source database before masking
if "all_sources" not in globals():
    raise ValueError(f"DF all_sources not found\n")

required_cols = ["source_id", "source_origin", "inchikey"]
missing_cols = [c for c in required_cols if c not in all_sources.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in all_sources: {missing_cols}\n")

# Category columns that must be IDENTICAL for two sources to be merged
# Previously, sources were merged only by origin + compound set, so e.g. an air freshener and a
# cleaning product (CPDat) containing the same single compound were merged and one category was lost
# The priors in INQUIRE_CSI part are computed from source_kind & source_general_category, so these must be kept
GROUP_META_COLS = ["source_kind", "source_general_category", "source_harmonized"]

def hash_set(elements):
    s = ",".join(sorted(set(elements)))
    return hashlib.md5(s.encode()).hexdigest()

# Score metadata completeness for representative selection
def metadata_score(row):
    score = 0
    metadata_cols = ["source_name", "source_kind", "source_general_category", "source_harmonized"]

    for col in metadata_cols:
        if col in row.index:
            value = row[col]
            if pd.notna(value):
                value = str(value).strip()
                if value != "":
                    score += 1
    return score


# Build source signatures
logging.info(f"Building source signatures")
logging.info(f" - Input rows: {len(all_sources):,}")
logging.info(f" - Unique source_ids: {all_sources['source_id'].nunique():,}")

source_signatures = (all_sources
    .dropna(subset=["inchikey"])
    .groupby(["source_origin", "source_id"], sort=False)["inchikey"]
    .agg(lambda x: hash_set(x)).reset_index(name="signature_hash"))

# Attach category metadata to each source signature
# Missing/empty values are replaced by "Unknown" (otherwise groupby would silently drop them)
source_group_meta = (all_sources
    .drop_duplicates(subset="source_id")[["source_id"] + GROUP_META_COLS]
    .copy())

for col in GROUP_META_COLS:
    source_group_meta[col] = source_group_meta[col].fillna("Unknown").astype(str).str.strip()
    source_group_meta.loc[source_group_meta[col] == "", col] = "Unknown"

source_signatures = source_signatures.merge(source_group_meta, on="source_id", how="left", validate="many_to_one")

logging.info(f" - Source signatures built: {len(source_signatures):,}")
minorsep()

# Build representative source lookup
logging.info(f"Building representative source lookup")
minorsep()

source_metadata = (all_sources.drop_duplicates(subset="source_id").copy())
source_metadata["metadata_score"] = (source_metadata.apply(metadata_score, axis=1))
source_metadata_lookup = (source_metadata.set_index("source_id"))

groups = {}

for row in tqdm(source_signatures.itertuples(index=False), total=len(source_signatures), desc="Building equivalence groups"):
    # group_key = (row.source_origin, row.signature_hash)
    group_key = (row.source_origin, row.source_kind, row.source_general_category, row.source_harmonized, row.signature_hash)

    if group_key not in groups:
        groups[group_key] = []

    groups[group_key].append(row.source_id)

# Build compound count lookup
minorsep()

n_sources = all_sources["source_id"].nunique()
compound_count_lookup = (all_sources.groupby("source_id")["inchikey"].nunique().to_dict())
compound_set_lookup = {}

for sid, grp in tqdm(all_sources.groupby("source_id"), total=n_sources, desc="Building compound lookups"):
    compound_set_lookup[sid] = set(grp["inchikey"].dropna())

minorsep()

metadata_score_lookup = (source_metadata[
        ["source_id", "metadata_score"]
    ]
    .drop_duplicates("source_id")
    .set_index("source_id")["metadata_score"]
    .to_dict())

# One rule for choosing the representative of each equivalence group:
#  - highest metadata completeness; ties → alphabetically first source_id
#    (previously the diagnostics used the alphabetically last id and the table the first id)
def choose_representative(sids):
    return min(sids, key=lambda sid: (-metadata_score_lookup.get(sid, -1), str(sid)))

# Diagnostic: equivalence class sizes
group_sizes = pd.Series([len(v) for v in groups.values()])

# Size of compound set represented by each equivalence class
eq_compound_sizes = []

for group_key, sids in tqdm(groups.items(), total=len(groups), desc="Equivalence diagnostics"):
    source_origin = group_key[0]
    h = group_key[-1]
    rep = choose_representative(sids)
    n_compounds = compound_count_lookup.get(rep, 0)

    eq_compound_sizes.append({"source_origin": source_origin,
        "source_harmonized": group_key[3],
        "hash": h,
        "group_size": len(sids),
        "n_compounds": n_compounds})
        
eq_stats = pd.DataFrame(eq_compound_sizes)

minorsep()
logging.info(f" - Equivalence groups with 1 compound: {(eq_stats['n_compounds'] == 1).sum():,}")
logging.info(f" - Equivalence groups with <=3 compounds: {(eq_stats['n_compounds'] <= 3).sum():,}")
logging.info(f" - Largest group compound count: {eq_stats.loc[eq_stats['group_size'].idxmax(), 'n_compounds']}\n")
logging.info(f" - Largest 20 equivalence classes:")
logging.info(f"{eq_stats.sort_values(['group_size', 'n_compounds'], ascending=False).head(20)}\n")
logging.info(f" - Equivalent groups (>1 source): {(group_sizes > 1).sum():,}")
logging.info(f" - Largest equivalence group: {group_sizes.max():,}")

# Inspect largest equivalence class
largest_key = max(groups, key=lambda h: len(groups[h]))
largest_group = groups[largest_key]

minorsep()
logging.info(f" - Largest group source origin: {largest_key[0]}")
logging.info(f" - Largest group category (source_harmonized): {largest_key[3]}")
logging.info(f" - Largest group representative: {choose_representative(largest_group)}")
logging.info(f" - First 5 source_ids in largest group:")
for sid in largest_group[:5]:
    logging.info(f"   - {sid}")
    
# Verify compound sets within largest equivalence group
sample_ids = largest_group[:5]
logging.info(f"\n - Number of unique compounds in 5 first groups:")
for sid in sample_ids:
    compounds = sorted(compound_set_lookup.get(sid, set()))
    logging.info(f"   - {sid}: {len(compounds):,} unique compounds")

# Compare first two members of largest group
if len(sample_ids) >= 2:
    s1 = sample_ids[0]
    s2 = sample_ids[1]
    c1 = compound_set_lookup.get(s1, set())
    c2 = compound_set_lookup.get(s2, set())

    logging.info(f"\n - Equality check between {s1} and {s2}: {c1 == c2}")
    logging.info(f" - Symmetric difference size: {len(c1.symmetric_difference(c2)):,}\n")

source_sizes = (all_sources.groupby("source_id")["inchikey"].nunique())

logging.info(f" - Sources with only 1 compound: {(source_sizes == 1).sum():,}")
logging.info(f" - Sources with <=3 compounds: {(source_sizes <= 3).sum():,}")
logging.info(f" - Median compounds/sources: {source_sizes.median():.1f}")
minorsep()

# build equivalence table
eq_rows = []
eq_summary_rows = []

# Fast source_id → source_name lookup (replaces filtering the full metadata table once per group)
source_name_lookup = source_metadata.set_index("source_id")["source_name"].to_dict()

# for (source_origin, h), sids in tqdm(groups.items(), total=len(groups), desc="Building equivalence table"):
for group_key, sids in tqdm(groups.items(), total=len(groups), desc="Building equivalence table"):
    source_origin, g_kind, g_category, g_harmonized, h = group_key
    rep = choose_representative(sids) # same rule as the old line above

    source_names = sorted({str(source_name_lookup.get(sid)) for sid in sids
                           if pd.notna(source_name_lookup.get(sid))})

    eq_summary_rows.append({"source_origin": source_origin,
        "source_kind": g_kind,
        "source_general_category": g_category,
        "source_harmonized": g_harmonized,
        "eqsetID": rep,
        "n_sources": len(sids),
        "source_ids": "; ".join(sorted(map(str, sids))),
        "source_names": "; ".join(sorted(source_names))})

    for sid in sids:
        eq_rows.append({"source_origin": source_origin,
            "source_harmonized": g_harmonized,
            "eqsetID": rep,
            "sourceID": sid})

eqsources = pd.DataFrame(eq_rows)
eqsource_summary = pd.DataFrame(eq_summary_rows)

minorsep()
largest_rep = (eqsource_summary.sort_values("n_sources", ascending=False).iloc[0])
logging.info(f" - Largest group representative chosen: {largest_rep['eqsetID']}")
logging.info(f" - Sources represented: {largest_rep['n_sources']:,}")
logging.info(f" - Representative source table rows: {len(eqsource_summary):,}")
logging.info(f" - Mean equivalent sources per group: {eqsource_summary['n_sources'].mean():.2f}")
logging.info(f" - Maximum equivalent sources per group: {eqsource_summary['n_sources'].max():,}\n")

# Reduce all_sources to unique profiles
unique_ids = eqsources["eqsetID"].unique()
all_sources = all_sources[all_sources["source_id"].isin(unique_ids)].copy()

# Reduction statistics
original_n = eqsources["sourceID"].nunique()
reduced_n = len(unique_ids)

reduction_pct = (100 * (original_n - reduced_n) / original_n
    if original_n > 0 else 0)

logging.info(f" - Equivalence sets: {reduced_n:,}")
logging.info(f" - Original products/sources: {original_n:,}")
logging.info(f" - Reduction: {reduction_pct:.1f}%\n")

# Per-source_origin reduction summary
reduction_summary = (eqsources
    .groupby("source_origin")
    .agg(original_sources=("sourceID", "nunique"),
        unique_sources=("eqsetID", "nunique"))
    .reset_index())

reduction_summary["reduction_pct"] = (100 * (reduction_summary["original_sources"]
        - reduction_summary["unique_sources"]) / reduction_summary["original_sources"])

logging.info(f" - Reduction by source origin:")
for _, row in reduction_summary.iterrows():
    logging.info(f"   - {row['source_origin']}: {row['original_sources']:,} → {row['unique_sources']:,} ({row['reduction_pct']:.1f}% reduction)")

minorsep()
logging.info(f" - eqsources rows: {len(eqsources):,}")
logging.info(f" - eqsource_summary rows: {len(eqsource_summary):,}")
minorsep()

# Save equivalence groups so every merged product can be traced back to its representative
STD_DIR = Path(DATA_DIR / "databases_standardized")
STD_DIR.mkdir(exist_ok=True)
eqsources.to_csv(STD_DIR / "equivalence_groups_members.csv", index=False)
eqsource_summary.to_csv(STD_DIR / "equivalence_groups_summary.csv", index=False)
logging.info(f"Saved equivalence group tables to: \n{STD_DIR}")
minorsep()

logging.info(f" - Aggregation runtime: {(time.time() - aggregation_start)/60:.1f} min")
minorsep()
logging.info(f"Done aggregating equivalent products/sources (same compound sets) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Show stats
majorsep()
logging.info(f"Showing stats for all_sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

def df_stats(df, df_name):
    stats = pd.DataFrame({
        'column': df.columns,
        'dtype': [df[c].dtype for c in df.columns],
        'n_unique': [df[c].nunique(dropna=True) for c in df.columns],
        'n_missing': [df[c].isna().sum() for c in df.columns],
        'pct_missing': [
            round(df[c].isna().mean() * 100, 2)
            for c in df.columns
        ]
    })

    print(f"\n{'='*80}")
    print(f"{df_name}")
    print(f"{'='*80}")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")

    display(stats)

# Run stats
df_stats(all_sources_0, "All sources before aggregation")
df_stats(all_sources, "All sources after aggregation")

minorsep()
logging.info(f"Done showing stats for all_sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Create a copy of aggregated all_sources
majorsep()
logging.info(f"Creating a copy of aggregated all_sources & saving it | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

all_sources_aggr = all_sources.copy()

# Create folder
AGGR_DIR = Path(DATA_DIR / "all_sources_aggregated")
AGGR_DIR.mkdir(exist_ok=True)

all_sources_aggr.to_csv(AGGR_DIR / f"all_sources_aggregated.csv", index=False)
logging.info(f" - CSV file saved | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

minorsep()
logging.info(f"Done creating a copy of aggregated all_sources & saving it | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Reshape sample data
majorsep()
logging.info(f"Reshaping sample matrix & building per-sample universes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

# Create copy of DF
dust_full_std = dust_full.copy()

# Remove sample rows without inchikeys
logging.info(f"Dropping rows without InChIKeys")
logging.info(f" - Sample dataset rows before InChIKey filtering: {len(dust_full_std):,}")

dust_full_std = dust_full_std.dropna(subset=["inchikey"])

logging.info(f" - Sample dataset rows after InChIKey filtering: {len(dust_full_std):,}")
minorsep()

# Drop unnecessary sample columns
# Specify columns to drop
dust_cols_drop = [
    '7S_CS_01',
    '7S_CS_02',
    '7S_CS_03',
    '7S_CS_04',
    'Blank_NIST',
]

# Switch: True = drop blank/QC columns (normal run); False = keep them
# (if kept, they are analyzed like samples in pt2 and plotted under their first two letters in pt3)
# DROP_QC_COLUMNS = False
DROP_QC_COLUMNS = True

if DROP_QC_COLUMNS:
    logging.info(f"Dropping unnecessary sample columns from DF\n")
    logging.info(f"Columns to be dropped:")
    for c in dust_cols_drop:
        if c in dust_full_std.columns:
            logging.info(f" - {c}")

    dust_cols_drop = [c for c in dust_cols_drop if c in dust_full_std.columns]
    dust_full_std = dust_full_std.drop(columns=dust_cols_drop)

    # safety check: no blank/QC columns may remain
    qc_left = [c for c in dust_full_std.columns if "blank" in str(c).lower() or str(c).upper().startswith("7S_CS")]
    if qc_left:
        logging.warning(f" - Blank/QC columns still present: {qc_left}")
    else:
        logging.info(f"\nNo blank/QC columns left")

    # Save standardized sample dataset, e.g., dust
    dust_full_std.to_csv(STD_DIR / "dust_standardized.csv", index=False) # rename CSV file accordingly; used in Pt2
    logging.info(f"\nSaved filtered sample matrix\n")
    logging.info(f"Dropped unnecessary sample columns (QC) from sample DF")
    minorsep()
    
else:
    logging.warning(f"DROP_QC_COLUMNS = False: blank/QC columns are kept and will be analyzed as samples")
    
    # Save standardized sample dataset, e.g., dust
    dust_full_std.to_csv(STD_DIR / "dust_standardized_qc_cols.csv", index=False) # rename CSV file accordingly; used in Pt2
    logging.info(f"\nSaved filtered sample matrix\n")
    logging.info(f"Kept unnecessary columns (QC)")
    minorsep()
        
# Create copy of DF
dust_reshaped = dust_full_std.copy()

logging.info(f"Reshaping sample matrix")

# Identify sample columns
start_col = list(dust_reshaped.columns).index("Concentration Type") + 1 # specify column after which sample columns begin
sample_cols = dust_reshaped.columns[start_col:]

# Build per-sample universes
sample_universes = {}

for s in sample_cols:
    present = dust_reshaped.loc[dust_reshaped[s] > 0, "inchikey"].dropna().unique()
    sample_universes[s] = sorted(present)

logging.info(f" - Samples: {len(sample_universes):,}")
example_sample = next(iter(sample_universes))
logging.info(f" - Example sample size (N features): {len(sample_universes[example_sample]):,}")

minorsep()
logging.info(f"Done reshaping sample matrix & building per-sample universes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Mask products/sources
majorsep()
logging.info(f"Masking products/sources (retain only compounds in sample matrix) | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

logging.info(f" - Initial product/source-compound rows (all_sources): {len(all_sources):,}")
n_uniq = all_sources["source_id"].nunique(dropna=True)
logging.info(f" - Initial product/source-compound (source_id): {n_uniq:,}")

sources_masked = all_sources[all_sources["inchikey"].isin(dust_reshaped["inchikey"])].copy() # sample dataset

# Second aggregation step after masking
#  - After removing compounds that never occur in samples, many sources become identical again
#    (e.g., two CPDat products that differ only in compounds not found in any sample)
#  - For the set cover these sources are indistinguishable in every sample, so merging them changes
#    nothing in the optimisation, but it avoids arbitrary picks between identical sources and
#    makes source-level validation metrics meaningful
#  - Sources are merged only if origin, source_kind, source_general_category & source_harmonized are identical
#  - Uses GROUP_META_COLS, hash_set() & metadata_score_lookup from the aggregation cell above
REAGGREGATE_AFTER_MASKING = True # set to False to switch this step off
# REAGGREGATE_AFTER_MASKING = False

if REAGGREGATE_AFTER_MASKING:
    minorsep()
    logging.info(f"Re-aggregating sources that became identical after masking (same origin & categories)")
    n_before_reagg = sources_masked["source_id"].nunique()

    # One row per source with its category key ("Unknown" for missing values)
    masked_keys = (sources_masked
        .drop_duplicates(subset="source_id")[["source_id", "source_origin"] + GROUP_META_COLS]
        .copy())

    for col in GROUP_META_COLS:
        masked_keys[col] = masked_keys[col].fillna("Unknown").astype(str).str.strip()
        masked_keys.loc[masked_keys[col] == "", col] = "Unknown"

    # Signature of each masked compound set
    masked_sigs = (sources_masked
        .groupby("source_id")["inchikey"]
        .agg(lambda x: hash_set(x))
        .reset_index(name="masked_hash"))

    masked_keys = masked_keys.merge(masked_sigs, on="source_id", how="inner", validate="one_to_one")

    # Representative = same rule as in the aggregation cell (highest metadata score; ties → first source_id)
    masked_keys["metadata_score"] = masked_keys["source_id"].map(metadata_score_lookup).fillna(-1)
    masked_keys = masked_keys.sort_values(["metadata_score", "source_id"], ascending=[False, True])

    # Own household products from different households are never merged
    # Their household is needed for the household-aware prior in the set cover part
    # ID 'FR01_PP04_HHP' → household 'FR01'; other origins → ""
    is_hhp = masked_keys["source_origin"] == "HOUSEHOLD_PRODUCTS"
    masked_keys["hh_household"] = ""
    masked_keys.loc[is_hhp, "hh_household"] = (masked_keys.loc[is_hhp, "source_id"].astype(str)
        .str.upper().str.extract(r"^([A-Z]{2}\d{2})_", expand=False)
        .fillna(masked_keys.loc[is_hhp, "source_id"].astype(str))) # unreadable ID → never merged
    masked_group_cols = ["source_origin"] + GROUP_META_COLS + ["hh_household", "masked_hash"]
    
    masked_keys["masked_eqsetID"] = (masked_keys
        .groupby(masked_group_cols, sort=False)["source_id"]
        .transform("first"))

    # Keep only the representatives
    masked_reps = set(masked_keys["masked_eqsetID"])
    sources_masked = sources_masked[sources_masked["source_id"].isin(masked_reps)].copy()
    n_after_reagg = sources_masked["source_id"].nunique()

    logging.info(f" - Sources before re-aggregation: {n_before_reagg:,}")
    logging.info(f" - Sources after re-aggregation: {n_after_reagg:,} ({percent(n_before_reagg - n_after_reagg, n_before_reagg)} reduction)")

    reagg_summary = (masked_keys
        .groupby("source_origin")
        .agg(before=("source_id", "nunique"), after=("masked_eqsetID", "nunique"))
        .reset_index())

    logging.info(f" - Re-aggregation by source origin:")
    for _, row in reagg_summary.iterrows():
        logging.info(f"   - {row['source_origin']}: {int(row['before']):,} → {int(row['after']):,}")
    minorsep()

# Remove products/sources with zero compounds
counts = sources_masked.groupby("source_id").size()
valid_sources = counts[counts > 0].index

sources_masked = sources_masked[sources_masked["source_id"].isin(valid_sources)]

logging.info(f" - Remaining product/source-compound rows (sources_masked): {len(sources_masked):,}")
logging.info(f" - Remaining products/sources (valid_sources): {len(valid_sources):,}")
logging.info(f" - Dropped product/source-compound rows: {len(all_sources)-len(sources_masked):,}")
logging.info(f" - Dropped products/sources: {n_uniq-len(valid_sources):,}")

minorsep()
logging.info(f"Done masking products/sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save masked products/sources
majorsep()
logging.info(f"Saving masked products/sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()

MASK_DIR = Path(DATA_DIR / "masked_sources")
MASK_DIR.mkdir(exist_ok=True)

# Save group membership so every merged source can be traced back to its representative
# delete in final
# (masked_keys[["masked_eqsetID", "source_id"] + masked_group_cols]
#     .sort_values(["masked_eqsetID", "source_id"])
#     .to_csv(MASK_DIR / "masked_equivalence_groups_members.csv", index=False))
# logging.info(f" - Saved masked equivalence groups")
if REAGGREGATE_AFTER_MASKING: # masked_keys only exists if the second aggregation was run
    (masked_keys[["masked_eqsetID", "source_id"] + masked_group_cols]
        .sort_values(["masked_eqsetID", "source_id"])
        .to_csv(MASK_DIR / "masked_equivalence_groups_members.csv", index=False))
    logging.info(f" - Saved masked equivalence groups")

sources_masked.to_csv(MASK_DIR / "sources_masked.csv", index=False)
logging.info(f" - Saved masked sources")

counts.rename("n_compound_rows").to_csv(MASK_DIR / "counts_masked.csv", index=True) # keep source_id in the file
logging.info(f" - Saved masked source counts")

pd.DataFrame({"source_id": valid_sources}).to_csv(MASK_DIR / "valid_sources_masked.csv", index=False)
logging.info(f" - Saved valid sources")
minorsep()

logging.info(f"Saved masked products/sources | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()


In [ ]:
# Save sample_universes
majorsep()
logging.info(f"Saving sample_universes | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

def align_dict_to_matrix(d):
    # collect all unique values
    all_keys = sorted({x for v in d.values() for x in v})

    # build aligned columns
    out = {}
    for col, vals in d.items():
        s = set(vals)
        out[col] = [x if x in s else "NA" for x in all_keys]

    #  add row index
    df = pd.DataFrame(out)
    df.insert(0, "inchikey", all_keys)

    return df

sample_universes_df = align_dict_to_matrix(sample_universes)
sample_universes_df.to_csv(MASK_DIR / "sample_universes.csv", index=False)

logging.info(f"Saved sample_universes to masked_sources folder | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
minorsep()
logging.info(f"Preprocessing pipeline complete | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
majorsep()
